# 倒傳遞演算法：讓電腦自己算梯度

對應參考書《深度學習入門：基於 Python 的理論與實現》第 4 章 4.3 至 4.4 節與第 5 章。
課本簡體版把倒傳遞演算法 backpropagation（BP）寫作「误差反向传播法」。

上一份《梯度下降法》的更新式是

$$w^{(k+1)} = w^{(k)} - \eta\,\frac{dL}{dw}$$

右邊那個 $dL/dw$ 一直是我們手推的。這一份回答：**梯度怎麼讓電腦自己算出來**。

示範函數和上一份相同：$L(w) = (w-3)^2$，起點 $w = 0$，數字可以互相核對。

| 段 | 做什麼 |
|---|---|
| 1 | 數值微分：只用導數的定義算梯度，把 $w = 0$ 拆開算一次 |
| 2 | 計算圖：乘法層與加法層，把課本買蘋果的例子跑一次正向、一次倒傳遞 |
| 3 | 回到 $L(w)$：自己寫兩個層，倒傳遞算出 −6 |
| 4 | 梯度確認：拿數值微分去驗倒傳遞，再故意寫錯一次看會怎樣 |
| 5 | 兩層網路：39,760 個參數，數值微分與倒傳遞比速度、也比答案 |
| 6 | 框架版：TensorFlow 的 GradientTape 做的就是倒傳遞 |

In [ ]:
import time

import numpy as np

np.set_printoptions(precision=6, suppress=True)

## 1. 數值微分

課本 4.3.1 節（書頁 95 至 96）在 $x$ 左右各取一點，算兩點連線的斜率，叫作中心差分：

$$\frac{df}{dx} \simeq \frac{f(x+h) - f(x-h)}{2h} \qquad h = 10^{-4}$$

不必推任何公式，只要會算 $f$ 就能算梯度。先把 $w = 0$ 拆開，每一個中間值都印出來。

In [ ]:
def loss(w):
    """L(w) = (w - 3)^2"""
    return (w - 3.0) ** 2


def numerical_diff(f, x, h=1e-4):
    """課本 4.3 的中心差分。"""
    return (f(x + h) - f(x - h)) / (2 * h)


h = 1e-4
print('L(0 + h)          = %.12f' % loss(0 + h))
print('L(0 - h)          = %.12f' % loss(0 - h))
print('分子 兩者相減     = %.12f' % (loss(h) - loss(-h)))
print('分母 2h           = %.12f' % (2 * h))
print('中心差分          = %.12f' % numerical_diff(loss, 0.0))
print('前向差分          = %.12f' % ((loss(h) - loss(0.0)) / h))
print('真正的導數 2(0-3) = %.12f' % (2 * (0 - 3)))

中心差分得到 $-6.000000000013$，和真正的 $-6$ 只差浮點數的捨入。
只取右邊一點的前向差分是 $-5.9999$，差了 $10^{-4}$：
對二次函數，前向差分的誤差剛好等於 $h$，中心差分沒有這一項。

數值微分的問題不在準不準，而在**貴**：每個參數都要把 $f$ 算兩次。
只有一個參數時無所謂，第 5 段會看到參數一多是什麼光景。

## 2. 計算圖：乘法層與加法層

課本 5.4 節把計算圖的每個節點寫成一個「層」，每一層只有兩個方法：
`forward` 算正向，`backward` 算倒傳遞。

兩條節點規則：

* **乘法**：上游傳來的值，乘以**另一邊**的輸入值。課本程式的註解寫「翻轉 x 和 y」，講的是同一件事
* **加法**：上游傳來的值**原封不動**往兩邊傳

In [ ]:
class MulLayer:
    """乘法層，課本 5.4.1。"""

    def __init__(self):
        self.x = None
        self.y = None

    def forward(self, x, y):
        self.x = x          # 正向時要記住輸入，倒傳遞會用到
        self.y = y
        return x * y

    def backward(self, dout):
        dx = dout * self.y  # 往 x 那邊乘 y
        dy = dout * self.x  # 往 y 那邊乘 x
        return dx, dy


class AddLayer:
    """加法層，課本 5.4.2。"""

    def forward(self, x, y):
        return x + y

    def backward(self, dout):
        return dout * 1, dout * 1

課本 5.1 節的例子：蘋果一個 100 圓，買 2 個，消費稅 1.1。

In [ ]:
apple, apple_num, tax = 100, 2, 1.1

mul_apple_layer = MulLayer()
mul_tax_layer = MulLayer()

# 正向
apple_price = mul_apple_layer.forward(apple, apple_num)
price = mul_tax_layer.forward(apple_price, tax)
print('正向：小計', apple_price, '／總價', price)

# 倒傳遞：呼叫順序和正向相反，起點是 1
dprice = 1
dapple_price, dtax = mul_tax_layer.backward(dprice)
dapple, dapple_num = mul_apple_layer.backward(dapple_price)
print('倒傳遞：蘋果價格', dapple, '／蘋果個數', dapple_num, '／消費稅', dtax)

課本的註解寫 `220` 與 `2.2 110 200`，實際印出來是 `220.00000000000003` 與
`110.00000000000001`。這是 1.1 在二進位裡存不精確造成的捨入，不是算錯。

**2.2 是什麼意思**：蘋果每漲 1 圓，總價就多 2.2 圓。正向重算一次就能驗：

In [ ]:
price_up = MulLayer().forward(MulLayer().forward(apple + 1, apple_num), tax)
print('蘋果漲 1 圓：總價 %.4f → %.4f，差 %.4f' % (price, price_up, price_up - price))
print('倒傳遞算的 dapple = %.4f' % dapple)

### 練習：蘋果加橘子

課本圖 5-17：蘋果 2 個、橘子 3 個，橘子每個 150 圓，兩者加總後乘消費稅。

**先在紙上**用兩條規則算出「橘子價格的導數」，寫下你的答案，再跑下一格對答案。

In [ ]:
orange, orange_num = 150, 3

mul_apple_layer = MulLayer()
mul_orange_layer = MulLayer()
add_apple_orange_layer = AddLayer()
mul_tax_layer = MulLayer()

# 正向
apple_price = mul_apple_layer.forward(apple, apple_num)
orange_price = mul_orange_layer.forward(orange, orange_num)
all_price = add_apple_orange_layer.forward(apple_price, orange_price)
price = mul_tax_layer.forward(all_price, tax)

# 倒傳遞
dall_price, dtax = mul_tax_layer.backward(1)
dapple_price, dorange_price = add_apple_orange_layer.backward(dall_price)
dorange, dorange_num = mul_orange_layer.backward(dorange_price)
dapple, dapple_num = mul_apple_layer.backward(dapple_price)

print('總價 %.1f' % price)
print('橘子價格的導數 dorange = %.1f' % dorange)

# 自己驗：橘子漲 1 圓再正向算一次
price_up = (apple * apple_num + (orange + 1) * orange_num) * tax
print('橘子漲 1 圓：總價 %.1f → %.1f，差 %.1f' % (price, price_up, price_up - price))

## 3. 回到 L(w)：自己寫兩個層

$L = (w-3)^2$ 拆成兩個節點：先減 3 得到 $t$，再平方得到 $L$。

| 節點 | 正向 | 局部導數 |
|---|---|---|
| 減 3 | $t = w - 3$ | $1$，和加法節點一樣原封不動往回傳 |
| 平方 | $L = t^2$ | $2t$，所以正向時要記住 $t$ |

鏈式法則把兩個局部導數乘起來：

$$\frac{dL}{dw} = \frac{dL}{dt}\cdot\frac{dt}{dw} = 2t \cdot 1 = 2(w-3)$$

In [ ]:
class SubConstLayer:
    """t = w - c。局部導數是 1。"""

    def __init__(self, c):
        self.c = c

    def forward(self, w):
        return w - self.c

    def backward(self, dout):
        return dout * 1


class SquareLayer:
    """L = t^2。局部導數是 2t。"""

    def __init__(self):
        self.t = None

    def forward(self, t):
        self.t = t
        return t ** 2

    def backward(self, dout):
        return dout * 2 * self.t


def backprop_grad(w, square=SquareLayer):
    """正向走一次、倒傳遞走一次，回傳 dL/dw。"""
    sub, sq = SubConstLayer(3.0), square()
    sq.forward(sub.forward(w))      # 正向
    dt = sq.backward(1.0)           # 倒傳遞從 1 開始
    return sub.backward(dt)


print('w = 0 時，倒傳遞算出 dL/dw =', backprop_grad(0.0))

得到 $-6$，和上一份教材手推的那一格一模一樣。

## 4. 梯度確認

課本 5.7.3 節：倒傳遞很快但容易寫錯，數值微分很慢但不會寫錯，所以拿慢的去驗快的。
這個動作叫梯度確認 gradient check。

**判準要先寫下來再跑**：兩者差的絕對值小於 $10^{-7}$ 才算通過。
不先定判準，看到什麼數字都會覺得合理。

In [ ]:
TOL = 1e-7
POINTS = [0.0, 0.6, 1.08, 1.464, 2.213568]   # 上一份教材梯度下降走過的點


def gradient_check(grad_fn, points=POINTS, tol=TOL):
    print('%-10s %-18s %-18s %-10s %s' % ('w', '數值微分', '倒傳遞', '|差|', '判定'))
    ok = True
    for w in points:
        gn = numerical_diff(loss, w)
        gb = grad_fn(w)
        d = abs(gn - gb)
        ok = ok and d < tol
        print('%-10.6f %-18.12f %-18.12f %-10.3e %s'
              % (w, gn, gb, d, '通過' if d < tol else '不通過'))
    print('整體判定：%s' % ('通過' if ok else '不通過，倒傳遞寫錯了'))
    return ok


gradient_check(backprop_grad)

### 故意寫錯一次

平方層的局部導數漏了 2，寫成 $t$。這是最常見的錯法之一。程式不會報任何錯。

In [ ]:
class SquareLayerBug(SquareLayer):
    def backward(self, dout):
        return dout * self.t        # 漏了 2


def backprop_grad_bug(w):
    return backprop_grad(w, square=SquareLayerBug)


print('w = 0 時，寫錯的倒傳遞算出 dL/dw =', backprop_grad_bug(0.0))
print()
gradient_check(backprop_grad_bug)

$-3$ 看起來是個很合理的數字。更麻煩的是，**拿它去做梯度下降，損失照樣會降**：

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams['axes.unicode_minus'] = False


def descend(grad_fn, w0=0.0, eta=0.1, steps=30):
    ws = [w0]
    for _ in range(steps):
        ws.append(ws[-1] - eta * grad_fn(ws[-1]))
    return np.array(ws)


good = descend(backprop_grad)
bad = descend(backprop_grad_bug)
print('正確版前 7 步：', np.round(good[:7], 6))
print('寫錯版前 7 步：', np.round(bad[:7], 6))

fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.plot(loss(good), 'o-', color='#1E7A45', ms=4, label='correct backprop: factor 0.8')
ax.plot(loss(bad), 's--', color='#C03030', ms=4, label='buggy backprop (missing 2): factor 0.9')
ax.set_xlabel('step'); ax.set_ylabel('L(w)')
ax.set_title('Both losses go down. Only the gradient check tells them apart.', fontsize=10)
ax.grid(alpha=0.3); ax.legend()
plt.tight_layout()
plt.show()

正確版的前 7 步是 0 → 0.6 → 1.08 → 1.464 → 1.7712 → 2.01696 → 2.213568，
和《梯度下降法》講義第二節的手算表一模一樣。

寫錯版每一步離 3 的距離剩九成，正確版剩八成。**兩條損失曲線都在降**，
只看損失有沒有下降，你不會發現它錯了。能抓到它的只有梯度確認。

## 5. 兩層網路：39,760 個參數

課本 4.5.2 節（書頁 115）訓練用的兩層網路是 784–50–10。參數個數：

| | 形狀 | 個數 |
|---|---|---|
| W1 | 784 × 50 | 39,200 |
| b1 | 50 | 50 |
| W2 | 50 × 10 | 500 |
| b2 | 10 | 10 |
| 合計 | | 39,760 |

下面依課本 5.6 至 5.7 節的結構寫一個最小版本：Affine 層、ReLU 層、Softmax-with-Loss 層。
課本用 MNIST 的前 3 筆資料做梯度確認；這裡改用固定種子的隨機資料，
不必下載資料集，而梯度確認的道理完全相同。

和課本有一處不同：交叉熵改用 log-sum-exp 寫法，不在 log 裡加 1e-7。理由寫在程式註解裡。

In [ ]:
class Affine:
    def __init__(self, W, b):
        self.W, self.b = W, b
        self.x = None
        self.dW = None
        self.db = None

    def forward(self, x):
        self.x = x
        return x @ self.W + self.b

    def backward(self, dout):
        self.dW = self.x.T @ dout
        self.db = dout.sum(axis=0)
        return dout @ self.W.T


class Relu:
    def __init__(self):
        self.mask = None

    def forward(self, x):
        self.mask = x <= 0
        out = x.copy()
        out[self.mask] = 0
        return out

    def backward(self, dout):
        dout = dout.copy()
        dout[self.mask] = 0
        return dout


class SoftmaxWithLoss:
    def __init__(self):
        self.y = None
        self.t = None

    def forward(self, x, t):
        # log-sum-exp 寫法：log softmax 直接算出來，不必在 log 裡加 1e-7。
        # 課本加 1e-7 是為了防止 log(0)，但那會讓損失和下面 (y - t)/N 這個梯度
        # 差一點點，梯度確認的差會被拉高到 1e-7 等級。練習 5 讓你自己試。
        z = x - x.max(axis=1, keepdims=True)
        log_y = z - np.log(np.exp(z).sum(axis=1, keepdims=True))
        self.y = np.exp(log_y)
        self.t = t
        return -np.sum(t * log_y) / x.shape[0]

    def backward(self, dout=1):
        return (self.y - self.t) / self.t.shape[0]


class TwoLayerNet:
    def __init__(self, input_size, hidden_size, output_size, seed=0, weight_init_std=0.01):
        rng = np.random.default_rng(seed)
        self.params = {
            'W1': weight_init_std * rng.standard_normal((input_size, hidden_size)),
            'b1': np.zeros(hidden_size),
            'W2': weight_init_std * rng.standard_normal((hidden_size, output_size)),
            'b2': np.zeros(output_size),
        }
        self.n_forward = 0      # 數一數正向一共跑了幾次

    def loss(self, x, t):
        self.n_forward += 1
        p = self.params
        self.a1 = Affine(p['W1'], p['b1'])
        self.r1 = Relu()
        self.a2 = Affine(p['W2'], p['b2'])
        self.last = SoftmaxWithLoss()
        return self.last.forward(self.a2.forward(self.r1.forward(self.a1.forward(x))), t)

    def gradient(self, x, t):
        """倒傳遞：正向 1 次、反向 1 次。"""
        self.loss(x, t)
        dout = self.last.backward()
        dout = self.a2.backward(dout)
        dout = self.r1.backward(dout)
        self.a1.backward(dout)
        return {'W1': self.a1.dW, 'b1': self.a1.db, 'W2': self.a2.dW, 'b2': self.a2.db}

    def numerical_gradient(self, x, t, h=1e-4):
        """數值微分：每個參數動一下，正向算 2 次。"""
        grads = {}
        for key, P in self.params.items():
            g = np.zeros_like(P)
            it = np.nditer(P, flags=['multi_index'])
            while not it.finished:
                i = it.multi_index
                keep = P[i]
                P[i] = keep + h
                fp = self.loss(x, t)
                P[i] = keep - h
                fm = self.loss(x, t)
                g[i] = (fp - fm) / (2 * h)
                P[i] = keep
                it.iternext()
            grads[key] = g
        return grads


net = TwoLayerNet(784, 50, 10)
print('參數個數 =', sum(P.size for P in net.params.values()))

rng = np.random.default_rng(1)
x_batch = rng.random((3, 784))                      # 3 筆，和課本一樣
t_batch = np.eye(10)[rng.integers(0, 10, size=3)]   # one-hot 標籤

兩種算法各跑一次，計時，也數正向跑了幾次。數值微分那一格要跑幾秒鐘。

In [ ]:
net.n_forward = 0
t0 = time.perf_counter()
grad_numerical = net.numerical_gradient(x_batch, t_batch)
t_num = time.perf_counter() - t0
n_num = net.n_forward

net.n_forward = 0
grad_backprop = net.gradient(x_batch, t_batch)
n_bp = net.n_forward
# 倒傳遞只跑一次太快，計時會被雜訊蓋過，所以跑 100 次取平均
t0 = time.perf_counter()
for _ in range(100):
    net.gradient(x_batch, t_batch)
t_bp = (time.perf_counter() - t0) / 100

print('數值微分：正向 %d 次，%.3f 秒' % (n_num, t_num))
print('倒傳遞  ：正向 %d 次、反向 1 次，平均 %.6f 秒' % (n_bp, t_bp))
print('倒傳遞快了約 %d 倍' % round(t_num / t_bp))

正向次數 79,520 = 39,760 × 2，一個不多一個不少。秒數依機器而定，倍數才是重點。

接著是課本 5.7.3 的梯度確認：每個參數兩種算法的差取絕對值，再取平均。

In [ ]:
for key in ('W1', 'b1', 'W2', 'b2'):
    diff = np.average(np.abs(grad_backprop[key] - grad_numerical[key]))
    print('%-3s 差的絕對值平均 = %.3e   %s' % (key, diff, '通過' if diff < TOL else '不通過'))

四組落在 $10^{-13}$ 到 $10^{-10}$ 之間，課本用 MNIST 得到的也是這個數量級。
換了資料或種子，數字會不一樣，數量級應該相同。

課本說得很清楚：差不會剛好是 0，因為電腦的浮點數精度有限；
寫對的話差會非常接近 0，**差很大就代表倒傳遞寫錯了**。

## 6. 框架版：GradientTape 就是倒傳遞

TensorFlow 的 `GradientTape` 把正向計算錄下來，再從最後面往回套鏈式法則，
做的就是前面那幾段的事，只是每一種運算的局部導數已經替你寫好了。

In [ ]:
import os

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf

print('TensorFlow', tf.__version__)

print('%-10s %-18s %-18s %s' % ('w', '手寫倒傳遞', 'GradientTape', '|差|'))
for w0 in POINTS:
    w = tf.Variable(w0, dtype=tf.float64)    # 用 float64，才能和手寫版逐位比
    with tf.GradientTape() as tape:
        t = w - 3.0
        L = t ** 2
    g_tf = float(tape.gradient(L, w))
    g_me = backprop_grad(w0)
    print('%-10.6f %-18.12f %-18.12f %.1e' % (w0, g_me, g_tf, abs(g_me - g_tf)))

兩邊逐位相同。

框架內建的運算不會漏掉那個 2，它們的局部導數都驗過了。
但你自己寫的層、自己寫的損失函數、自訂的梯度就沒有人替你驗——
那時候能依靠的，就是第 4 段那張表。

## 練習

1. 把 `SubConstLayer(3.0)` 改成減 5，倒傳遞的結果會變嗎？為什麼？先預測再跑。
2. 寫一個 `CubeLayer`，正向是 $t^3$。它的 `backward` 要乘什麼？用 `gradient_check` 驗你寫的對不對。
3. 第 5 段把 `h` 改成 `1e-2` 再做一次梯度確認，差會變大還是變小？改成 `1e-10` 呢？
4. 第 5 段的 `Relu.backward` 故意漏掉 `dout[self.mask] = 0` 那一行，梯度確認抓得到嗎？
5. 把 `SoftmaxWithLoss.forward` 改回課本的 `np.log(self.y + 1e-7)`，再做一次梯度確認。
   哪一組參數的差變最大？為什麼一個 $10^{-7}$ 就看得出來？